# CyberShield AI - AI-Based Cyber Threat Detection System
### End-to-End Machine Learning & Deep Learning Pipeline for Intrusion Detection
---
**Project Objectives:**
1. Ingest & preprocess NSL-KDD and network traffic telemetry.
2. Train supervised multi-class ML models (Random Forest, SVM, Decision Tree, Logistic Regression).
3. Build and evaluate Deep Neural Networks (MLP) with Dropout & ReLU.
4. Develop an Autoencoder for zero-day anomaly detection.
5. Compute 0-100 Cyber Threat Risk Scores and Explainable AI attributions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.append(os.path.abspath('..'))

from utils.dataset_loader import load_benchmark_dataset
from utils.preprocessor import CyberDataPreprocessor
from models.ml_models import CyberMLModelHub
from models.deep_learning import CyberDeepNeuralNetwork
from models.autoencoder import CyberAutoencoder
from models.explainable_ai import ThreatExplainer
from utils.severity_engine import calculate_risk_score

## 1. Load NSL-KDD Dataset

In [ ]:
df = load_benchmark_dataset('NSL-KDD')
print(f'Loaded {len(df):,} network records with {df.shape[1]} features.')
df.head()

## 2. Preprocess Data & Feature Engineering

In [ ]:
pre = CyberDataPreprocessor(scaling_method='StandardScaler', n_features=20)
data = pre.fit_transform(df, target_col='attack_category', select_features=True)

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
classes = data['target_classes']

print(f'Training shape: {X_train.shape}, Testing shape: {X_test.shape}')
print(f'Target classes: {classes}')

## 3. Train Machine Learning Classifiers (Random Forest, SVM, Decision Tree, Logistic Regression)

In [ ]:
hub = CyberMLModelHub()
results = {}

for model_name in ['Random Forest', 'Decision Tree', 'Logistic Regression']:
    print(f'Training {model_name}...')
    res = hub.train_and_evaluate(model_name, X_train, y_train, X_test, y_test, class_names=classes)
    results[model_name] = res
    print(f"{model_name} -> Accuracy: {res['accuracy']*100:.2f}%, F1-Score: {res['f1']*100:.2f}%")

## 4. Train Deep Neural Network (MLP)

In [ ]:
dnn = CyberDeepNeuralNetwork(input_dim=X_train.shape[1], hidden_layers=(128, 64), num_classes=len(classes), epochs=30)
history = dnn.fit(X_train, y_train, X_test, y_test)
dnn_eval = dnn.evaluate(X_test, y_test, class_names=classes)
print(f"Deep Neural Network Accuracy: {dnn_eval['accuracy']*100:.2f}%, F1: {dnn_eval['f1']*100:.2f}%")

## 5. Unsupervised Autoencoder Anomaly Detection

In [ ]:
normal_idx = classes.index('Normal') if 'Normal' in classes else 0
X_train_normal = X_train[y_train == normal_idx]
y_test_bin = (y_test != normal_idx).astype(int)

ae = CyberAutoencoder(input_dim=X_train.shape[1], latent_dim=6, epochs=30)
ae.fit(X_train_normal)
ae_eval = ae.evaluate(X_test, y_test_bin)
print(f"Autoencoder Anomaly Detection ROC-AUC: {ae_eval['roc_auc']:.3f}")

## 6. Threat Risk Engine & Explainable AI (XAI)

In [ ]:
explainer = ThreatExplainer(feature_names=data['feature_names'])
sample_idx = 5
sample_packet = X_test[sample_idx]
pred_cls = classes[y_test[sample_idx]]

explanation = explainer.explain_sample(sample_packet, data['feature_names'], pred_cls, confidence=0.96)
print('Narrative Explanation:')
print(explanation['narrative'])
print('\nTop Contributing Features:')
for f in explanation['top_features'][:4]:
    print(f" - {f['feature']}: {f['contribution_pct']}% impact")